In [1]:
# ==============================================================================
# Cell 1: Environment Setup & Library Imports
# ==============================================================================
import os
import sys
import glob
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Dropout, Conv1D, GlobalAveragePooling1D, Lambda
from tensorflow.keras.regularizers import l2

from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_curve, auc, f1_score
from sklearn.utils import resample
from scipy.spatial.distance import cdist

# Set global random seed for exact reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Detect runtime environment (Google Colab vs. Local IDE)
candidate_dirs = [
    os.path.abspath(r"e:/Obaidul Haque/Alpha Band Analysis/Alpha Band Staging"),
    os.path.abspath("./Alpha Band Analysis/Alpha Band Staging"),
    os.path.abspath("."),
    "/content/drive/MyDrive/Alpha Band Analysis/Alpha Band Staging"
]
base_dir = None
for d in candidate_dirs:
    if os.path.exists(os.path.join(d, "Preprocessed Data", "testing")):
        base_dir = d
        break
if base_dir is None:
    base_dir = candidate_dirs[0]

# Define relative data and model directories
data_testing_dir = os.path.join(base_dir, "Preprocessed Data", "testing")
data_training_dir = os.path.join(base_dir, "Preprocessed Data", "training")
models_dir = os.path.join(base_dir, "Trained Models")
output_dir = os.path.join(base_dir, "..", "Results_and_Figures")
os.makedirs(output_dir, exist_ok=True)

print(f"Base Directory: {base_dir}")
print(f"Testing Data Path: {data_testing_dir}")
print(f"Trained Models Path: {models_dir}")
print(f"Outputs will be saved to: {output_dir}")


Mounted at /content/drive
Running in Google Colab environment. Mounted Google Drive.
Testing Data Path: /content/drive/MyDrive/Alpha Band Analysis/Alpha Band Staging/Preprocessed Data/testing
Trained Models Path: /content/drive/MyDrive/Alpha Band Analysis/Alpha Band Staging/Trained Models
Outputs will be saved to: /content/drive/MyDrive/Alpha Band Analysis/Alpha Band Staging/Extended_Results_Outputs


In [2]:
# ==============================================================================
# Cell 2: Derivation Topologies, Biomarker Names & Helper Utilities
# ==============================================================================

# Standard 23 Bipolar EEG Derivations (CHB-MIT Longitudinal Bipolar Montage)
CHANNELS = [
    'FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1',
    'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
    'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2',
    'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2',
    'FZ-CZ', 'CZ-PZ',
    'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1'
]
CHANNEL_INDICES = {ch: i for i, ch in enumerate(CHANNELS)}
NUM_CHANNELS = len(CHANNELS)  # 23

# 22 Extracted Features (17 Time-Domain + 5 Frequency-Domain)
FEATURE_NAMES = [
    "Mean", "Median", "Variance", "Std Deviation", "Root Mean Square (RMS)",
    "Peak-to-Peak Amp", "Max Amp", "Min Amp", "Skewness", "Kurtosis",
    "Line Length", "Zero Crossings", "Energy", "Shannon Entropy",
    "Hjorth Activity", "Hjorth Mobility", "Hjorth Complexity",
    "Total Power", "Peak Frequency", "Mean Frequency",
    "Spectral Entropy", "Spectral Edge Frequency"
]
NUM_FEATURES = len(FEATURE_NAMES)  # 22
TOTAL_FEATURES = NUM_CHANNELS * NUM_FEATURES  # 506

# 5 Operational Clinical Staging Categories
CATEGORIES = ['Normal', 'Pre-Ictal', 'Ictal', 'Inter-Ictal', 'Post-Ictal']
CLASS_COLORS = ['#2ca02c', '#ff7f0e', '#d62728', '#9467bd', '#1f77b4']

def get_file_paths(folder_path):
    """Finds all CSV files in a directory sorted for deterministic evaluation."""
    paths = glob.glob(os.path.join(folder_path, "*.csv"))
    return sorted([p for p in paths if os.path.isfile(p)])

def get_max_occurring_from_last_n(arr, n=5):
    """Finds the maximum occurring class label from the last n segments."""
    if not arr or n <= 0:
        return None, 0
    last_n_items = arr[-n:]
    flattened_items = [item[0] for item in last_n_items if isinstance(item, (list, np.ndarray)) and len(item) > 0]
    if not flattened_items:
        return None, 0
    item_counts = Counter(flattened_items)
    most_common_item = item_counts.most_common(1)
    return most_common_item[0] if most_common_item else (None, 0)

def create_sliding_window(features_2d, labels_raw, window_size=10, stride=1):
    """
    Constructs multi-segment sliding sequence representations.
    Input features_2d: (num_segments, 506)
    Returns: window_data (num_windows, 10, 506), window_labels (num_windows,)
    """
    num_segments = features_2d.shape[0]
    if num_segments < window_size:
        return np.empty((0, window_size, features_2d.shape[1])), np.empty((0,))

    windows_data = []
    window_presence_labels = []

    for i in range(0, num_segments - window_size + 1, stride):
        win_data = features_2d[i:i + window_size]
        win_labels = labels_raw[i:i + window_size]
        windows_data.append(win_data)
        
        # Majority vote from the last 5 segments of the sequence
        max_item, _ = get_max_occurring_from_last_n([[int(win_labels[j][0])] for j in range(len(win_labels))], n=5)
        window_presence_labels.append(max_item)

    return np.array(windows_data, dtype=np.float32), np.array(window_presence_labels, dtype=np.int32)


In [3]:
# ==============================================================================
# Cell 3: Clinical Transition Distance Topology Matrix (D_trans) & Metrics
# ==============================================================================

def build_transition_distance_matrix():
    """
    Builds the 5x5 clinical state transition distance matrix based on the
    physiological progression topology defined in the manuscript:
    Normal (0) <-> Pre-Ictal (1) <-> Ictal (2) <-> Inter-Ictal (3)
                                              <-> Post-Ictal (4) <-> Normal (0)
    """
    # Shortest path transition distance matrix
    D_trans = np.array([
        [0.0, 1.0, 2.0, 3.0, 1.0],  # 0: Normal
        [1.0, 0.0, 1.0, 2.0, 2.0],  # 1: Pre-Ictal
        [2.0, 1.0, 0.0, 1.0, 1.0],  # 2: Ictal
        [3.0, 2.0, 1.0, 0.0, 2.0],  # 3: Inter-Ictal
        [1.0, 2.0, 1.0, 2.0, 0.0],  # 4: Post-Ictal
    ], dtype=np.float32)
    return D_trans

D_TRANS = build_transition_distance_matrix()

def compute_transition_metrics(y_true, y_pred, D_trans=D_TRANS):
    """
    Computes exact accuracy, Clinical Transition Agreement (CTA <= 1 step %),
    and Mean Transition Stage Deviation (MTSD).
    """
    y_true = np.array(y_true, dtype=int)
    y_pred = np.array(y_pred, dtype=int)
    N = len(y_true)
    if N == 0:
        return 0.0, 0.0, 0.0, 0.0

    # 1. Exact Accuracy
    exact_acc = np.mean(y_true == y_pred) * 100.0

    # 2. Transition Distances
    distances = D_trans[y_true, y_pred]
    mtsd = np.mean(distances)

    # 3. Clinical Transition Agreement (CTA): percentage within <= 1 transition step
    cta = np.mean(distances <= 1.0) * 100.0

    # 4. Macro F1
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

    return exact_acc, cta, mtsd, macro_f1

print("Transition Distance Matrix (D_trans):")
print(pd.DataFrame(D_TRANS, index=CATEGORIES, columns=CATEGORIES))


Transition Distance Matrix (D_trans):
             Normal  Pre-Ictal  Ictal  Inter-Ictal  Post-Ictal
Normal          0.0        1.0    2.0          3.0         1.0
Pre-Ictal       1.0        0.0    1.0          2.0         2.0
Ictal           2.0        1.0    0.0          1.0         1.0
Inter-Ictal     3.0        2.0    1.0          0.0         2.0
Post-Ictal      1.0        2.0    1.0          2.0         0.0


In [4]:
# ==============================================================================
# Cell 4: Recording-Level Independent Test Cohort Loader
# ==============================================================================

test_csv_files = get_file_paths(data_testing_dir)
print(f"Identified {len(test_csv_files)} recording files in test directory.")

# Session metadata dictionary (21 sessions across 14 patients)
recording_data = {}

for file_path in tqdm(test_csv_files, desc="Loading Test Sessions"):
    session_id = os.path.basename(file_path).replace(".csv", "")
    df = pd.read_csv(file_path, header=None)
    
    # Last column is the label (0 to 4)
    features_2d = df.iloc[:, :-1].values
    labels_raw = df.iloc[:, -1].values.reshape(-1, 1)

    # Generate sliding window sequence tensors
    X_win, y_win = create_sliding_window(features_2d, labels_raw, window_size=10, stride=1)
    
    if len(X_win) > 0:
        recording_data[session_id] = {
            "file_path": file_path,
            "X": X_win,
            "y": y_win,
            "num_windows": len(X_win),
            "label_dist": Counter(y_win)
        }

print(f"\nSuccessfully processed {len(recording_data)} valid test recording sessions.")
total_test_windows = sum(v["num_windows"] for v in recording_data.values())
print(f"Total Test Windows across cohort: {total_test_windows}")


SyntaxError: unterminated f-string literal (detected at line 31) (2341757803.py, line 31)

In [ ]:
# ==============================================================================
# Cell 5: Model Loading (Bi-LSTM, Stacked LSTM, TCN1, TCN2)
# ==============================================================================

def create_BiLSTM_model(input_shape=(10, 506), learning_rate=1e-3, dropout_rate=0.5, l2_reg=1e-5, units=64):
    inputs = tf.keras.Input(shape=input_shape)
    x = Bidirectional(LSTM(units, return_sequences=True, kernel_regularizer=l2(l2_reg)))(inputs)
    x = Dropout(dropout_rate)(x)
    x = Bidirectional(LSTM(units, return_sequences=False, kernel_regularizer=l2(l2_reg)))(x)
    x = Dropout(dropout_rate)(x)
    output = Dense(5, activation='softmax')(x)
    model = Model(inputs=inputs, outputs=output, name="Bi-LSTM")
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Primary model: Bi-LSTM
bilstm_path = os.path.join(models_dir, "Alpha_band_BiLSTM.keras")
model_bilstm = create_BiLSTM_model(input_shape=(10, 506))

if os.path.exists(bilstm_path):
    try:
        model_bilstm.load_weights(bilstm_path)
        print(f"Loaded trained Bi-LSTM weights successfully from: {bilstm_path}")
    except Exception as e:
        print(f"Note: Error loading weights ({e}). Initialized fresh model architecture.")
else:
    print(f"Model file not found at {bilstm_path}. Running with initialized architecture.")

model_bilstm.summary()


In [ ]:
# ==============================================================================
# Cell 6: Table 10 - Recording-Level Staging Performance Evaluation
# ==============================================================================

table_10_rows = []
all_y_true = []
all_y_pred = []
all_session_tags = []

for session_idx, (session_id, data_dict) in enumerate(recording_data.items(), start=1):
    X_sess = data_dict["X"]
    y_sess = data_dict["y"]

    y_probs = model_bilstm.predict(X_sess, verbose=0)
    y_pred = np.argmax(y_probs, axis=1)

    exact_acc, cta, mtsd, macro_f1 = compute_transition_metrics(y_sess, y_pred)

    table_10_rows.append({
        "Session": f"Session {session_idx:02d}",
        "Recording": session_id,
        "Windows": len(y_sess),
        "Exact Acc. (%)": round(exact_acc, 2),
        "CTA (<=1 Step %)": round(cta, 2),
        "MTSD (Steps)": round(mtsd, 4),
        "Macro F1": round(macro_f1, 4)
    })

    all_y_true.extend(y_sess)
    all_y_pred.extend(y_pred)
    all_session_tags.extend([session_id] * len(y_sess))

df_table_10 = pd.DataFrame(table_10_rows)

# Compute Cohort Mean & Std Dev
mean_row = {
    "Session": "Cohort Mean ± SD",
    "Recording": f"N={len(table_10_rows)} Sessions",
    "Windows": sum(r["Windows"] for r in table_10_rows),
    "Exact Acc. (%)": f"{df_table_10['Exact Acc. (%)'].mean():.2f} ± {df_table_10['Exact Acc. (%)'].std():.2f}",
    "CTA (<=1 Step %)": f"{df_table_10['CTA (<=1 Step %)'].mean():.2f} ± {df_table_10['CTA (<=1 Step %)'].std():.2f}",
    "MTSD (Steps)": f"{df_table_10['MTSD (Steps)'].mean():.4f} ± {df_table_10['MTSD (Steps)'].std():.4f}",
    "Macro F1": f"{df_table_10['Macro F1'].mean():.4f} ± {df_table_10['Macro F1'].std():.4f}"
}

print("\n" + "="*80)
print("TABLE 10: Recording-Level Staging Performance of Bi-LSTM Across 21 Test Sessions")
print("="*80)
print(df_table_10.to_string(index=False))
print("-" * 80)
for k, v in mean_row.items():
    print(f"{k}: {v}")
print("="*80)

# Save Table 10 to CSV
csv_table_10_path = os.path.join(output_dir, "Table_10_Recording_Level_Staging_Performance.csv")
df_table_10.to_csv(csv_table_10_path, index=False)
print(f"Saved Table 10 to: {csv_table_10_path}")


In [ ]:
# ==============================================================================
# Cell 7: Patient-Clustered Bootstrap Resampling (2,000 Iterations, 95% CI)
# ==============================================================================

N_BOOTSTRAP = 2000
unique_sessions = list(recording_data.keys())
n_sessions = len(unique_sessions)

boot_acc, boot_cta, boot_mtsd, boot_f1 = [], [], [], []

print(f"Executing {N_BOOTSTRAP} Patient-Clustered Bootstrap Iterations...")
for b in tqdm(range(N_BOOTSTRAP), desc="Bootstrap Resampling"):
    sampled_sessions = np.random.choice(unique_sessions, size=n_sessions, replace=True)
    
    b_true, b_pred = [], []
    for sess in sampled_sessions:
        b_true.extend(recording_data[sess]["y"])
        # Use recorded predictions
        b_pred.extend(model_bilstm.predict(recording_data[sess]["X"], verbose=0).argmax(axis=1))
        
    acc, cta, mtsd, f1 = compute_transition_metrics(b_true, b_pred)
    boot_acc.append(acc)
    boot_cta.append(cta)
    boot_mtsd.append(mtsd)
    boot_f1.append(f1)

def get_ci(arr):
    return np.percentile(arr, 2.5), np.percentile(arr, 97.5)

ci_summary = pd.DataFrame([
    {"Metric": "Exact Accuracy (%)", "Mean": np.mean(boot_acc), "95% CI Lower": get_ci(boot_acc)[0], "95% CI Upper": get_ci(boot_acc)[1]},
    {"Metric": "CTA (<=1 Step %)", "Mean": np.mean(boot_cta), "95% CI Lower": get_ci(boot_cta)[0], "95% CI Upper": get_ci(boot_cta)[1]},
    {"Metric": "MTSD (Steps)", "Mean": np.mean(boot_mtsd), "95% CI Lower": get_ci(boot_mtsd)[0], "95% CI Upper": get_ci(boot_mtsd)[1]},
    {"Metric": "Macro F1-Score", "Mean": np.mean(boot_f1), "95% CI Lower": get_ci(boot_f1)[0], "95% CI Upper": get_ci(boot_f1)[1]},
])

print("\n" + "="*80)
print("Patient-Clustered Bootstrap Confidence Intervals (2,000 Iterations):")
print("="*80)
print(ci_summary.to_string(index=False))
ci_summary.to_csv(os.path.join(output_dir, "Bootstrap_95CI_Staging_Summary.csv"), index=False)


In [ ]:
# ==============================================================================
# Cell 8: Explainable AI (XAI) Framework - Integrated Gradients & Input x Gradient
# ==============================================================================

@tf.function
def compute_gradients_at_interpolation(model, interpolated_inputs, target_class_idx):
    with tf.GradientTape() as tape:
        tape.watch(interpolated_inputs)
        preds = model(interpolated_inputs)
        target_logits = preds[:, target_class_idx]
    grads = tape.gradient(target_logits, interpolated_inputs)
    return grads

def integrated_gradients(model, input_sequence, target_class_idx=None, m_steps=50, baseline=None):
    """
    Computes Integrated Gradients attribution for sequence input (10, 506).
    """
    if baseline is None:
        baseline = np.zeros_like(input_sequence)
        
    if target_class_idx is None:
        preds = model(np.expand_dims(input_sequence, axis=0))
        target_class_idx = int(np.argmax(preds[0]))

    # Generate alphas in [0, 1]
    alphas = np.linspace(0.0, 1.0, m_steps + 1)[:, np.newaxis, np.newaxis]
    interpolated_inputs = baseline + alphas * (input_sequence - baseline)
    interpolated_inputs_tensor = tf.convert_to_tensor(interpolated_inputs, dtype=tf.float32)

    # Compute gradients along linear path
    grads = compute_gradients_at_interpolation(model, interpolated_inputs_tensor, target_class_idx)
    avg_grads = tf.reduce_mean(grads, axis=0)
    
    # Path integral approximation
    integrated_grad = (input_sequence - baseline) * avg_grads.numpy()
    return integrated_grad

def decompose_attributions(attribution_matrix):
    """
    Decomposes (10, 506) attribution into:
      1. Derivation attributions (23 channels)
      2. Feature attributions (22 features)
    """
    # Attribution is (10 timesteps, 506 features)
    # Reshape to (10 timesteps, 23 channels, 22 features)
    abs_attr = np.abs(attribution_matrix).reshape(10, NUM_CHANNELS, NUM_FEATURES)
    
    # Derivation saliency: sum across time and features
    channel_saliency = np.sum(abs_attr, axis=(0, 2))
    channel_saliency_pct = (channel_saliency / np.sum(channel_saliency)) * 100.0
    
    # Feature saliency: sum across time and channels
    feature_saliency = np.sum(abs_attr, axis=(0, 1))
    feature_saliency_pct = (feature_saliency / np.sum(feature_saliency)) * 100.0
    
    return channel_saliency_pct, feature_saliency_pct

print("XAI Engine initialized successfully with Integrated Gradients and Attribution Decomposition.")


In [ ]:
# ==============================================================================
# Cell 9: Table 11 - Recording-Level Top XAI Attributions across 21 Test Sessions
# ==============================================================================

table_11_rows = []
cohort_channel_attributions = np.zeros(NUM_CHANNELS)
cohort_feature_attributions = np.zeros(NUM_FEATURES)

print("Computing Recording-Level XAI Attributions for Table 11...")
for session_idx, (session_id, data_dict) in enumerate(tqdm(recording_data.items(), desc="XAI Saliency"), start=1):
    X_sess = data_dict["X"]
    
    # Compute average attribution over representative samples in the session
    n_samples = min(len(X_sess), 15)
    sample_indices = np.linspace(0, len(X_sess)-1, n_samples, dtype=int)
    
    session_ch_attr = np.zeros(NUM_CHANNELS)
    session_feat_attr = np.zeros(NUM_FEATURES)
    
    for idx in sample_indices:
        sample_seq = X_sess[idx]
        ig_attr = integrated_gradients(model_bilstm, sample_seq, m_steps=20)
        ch_pct, feat_pct = decompose_attributions(ig_attr)
        session_ch_attr += ch_pct
        session_feat_attr += feat_pct
        
    session_ch_attr /= n_samples
    session_feat_attr /= n_samples
    
    cohort_channel_attributions += session_ch_attr
    cohort_feature_attributions += session_feat_attr
    
    # Top 2 Derivations
    top_ch_indices = np.argsort(session_ch_attr)[::-1][:2]
    top_ch1 = CHANNELS[top_ch_indices[0]]
    top_ch2 = CHANNELS[top_ch_indices[1]]
    
    # Top 2 Biomarkers
    top_feat_indices = np.argsort(session_feat_attr)[::-1][:2]
    top_feat1 = FEATURE_NAMES[top_feat_indices[0]]
    top_feat2 = FEATURE_NAMES[top_feat_indices[1]]
    
    table_11_rows.append({
        "Session": f"Session {session_idx:02d} ({session_id})",
        "Top Derivation 1": top_ch1,
        "Top Derivation 2": top_ch2,
        "Top Biomarker 1": top_feat1,
        "Top Biomarker 2": top_feat2
    })

df_table_11 = pd.DataFrame(table_11_rows)
cohort_channel_attributions /= len(recording_data)
cohort_feature_attributions /= len(recording_data)

print("\n" + "="*80)
print("TABLE 11: Recording-Level Staging Performance and Top XAI Attributions (Bi-LSTM)")
print("="*80)
print(df_table_11.to_string(index=False))

csv_table_11_path = os.path.join(output_dir, "Table_11_Top_XAI_Attributions_BiLSTM.csv")
df_table_11.to_csv(csv_table_11_path, index=False)
print(f"\nSaved Table 11 to: {csv_table_11_path}")


In [ ]:
# ==============================================================================
# Cell 10: Cohort-Level Saliency Rankings (Figure 16 Equivalent)
# ==============================================================================

plt.figure(figsize=(16, 6), dpi=300)

# Panel 1: Derivation Saliency Ranking
plt.subplot(1, 2, 1)
sorted_ch_idx = np.argsort(cohort_channel_attributions)[::-1]
ch_df = pd.DataFrame({
    "Channel": [CHANNELS[i] for i in sorted_ch_idx],
    "Saliency": cohort_channel_attributions[sorted_ch_idx]
})
sns.barplot(data=ch_df, x="Saliency", y="Channel", palette="viridis")
plt.title("Bipolar Derivation Saliency Ranking (23 Channels)", fontsize=13, fontweight='bold')
plt.xlabel("Integrated Gradients Attribution (%)", fontsize=11)
plt.ylabel("Derivation", fontsize=11)

# Panel 2: Feature Saliency Ranking
plt.subplot(1, 2, 2)
sorted_feat_idx = np.argsort(cohort_feature_attributions)[::-1]
feat_df = pd.DataFrame({
    "Feature": [FEATURE_NAMES[i] for i in sorted_feat_idx],
    "Saliency": cohort_feature_attributions[sorted_feat_idx]
})
sns.barplot(data=feat_df, x="Saliency", y="Feature", palette="magma")
plt.title("Electrophysiological Biomarker Saliency Ranking (22 Features)", fontsize=13, fontweight='bold')
plt.xlabel("Integrated Gradients Attribution (%)", fontsize=11)
plt.ylabel("Feature", fontsize=11)

plt.tight_layout()
fig16_path = os.path.join(output_dir, "Figure_16_Cohort_Derivation_and_Feature_Saliency.png")
plt.savefig(fig16_path, dpi=300)
plt.show()
print(f"Saved Cohort Saliency Plot to: {fig16_path}")


In [ ]:
# ==============================================================================
# Cell 11: Patient 12 Case Study Deep Dive (Figure 17 / Figure 13)
# ==============================================================================

patient12_key = None
for k in recording_data.keys():
    if "chb12_08" in k or "chb12" in k:
        patient12_key = k
        break

if patient12_key is None:
    patient12_key = list(recording_data.keys())[min(9, len(recording_data)-1)]

print(f"Generating 3-Panel Deep Dive for Case Study Patient: {patient12_key}...")
p12_X = recording_data[patient12_key]["X"]
p12_y = recording_data[patient12_key]["y"]
p12_probs = model_bilstm.predict(p12_X, verbose=0)

# Time axis (each window is 6s stride)
time_axis_min = np.arange(len(p12_y)) * 6.0 / 60.0

fig, axes = plt.subplots(3, 1, figsize=(15, 12), dpi=300)

# Panel A: State-Dependent Evolution of Posterior Probabilities
for class_idx in range(5):
    axes[0].plot(time_axis_min, p12_probs[:, class_idx], label=CATEGORIES[class_idx], color=CLASS_COLORS[class_idx], lw=1.8)
axes[0].set_title(f"Panel A: Temporal Evolution of Posterior Probabilities for Patient 12 ({patient12_key})", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Recording Time (Minutes)", fontsize=11)
axes[0].set_ylabel("Posterior Probability", fontsize=11)
axes[0].set_ylim([-0.05, 1.05])
axes[0].legend(loc="upper right", frameon=True, ncol=5)
axes[0].grid(True, alpha=0.3)

# Panel B: Feature-Level Attribution Dynamics Across Clinical States
key_features = ["Line Length", "Skewness", "Median", "Hjorth Activity", "Hjorth Complexity", "Hjorth Mobility", "Zero Crossings"]
feat_indices = [FEATURE_NAMES.index(f) for f in key_features if f in FEATURE_NAMES]

state_feature_attrs = np.zeros((5, len(feat_indices)))
for s in range(5):
    s_idx = np.where(p12_y == s)[0]
    if len(s_idx) > 0:
        s_samples = s_idx[:min(len(s_idx), 10)]
        attrs = []
        for idx in s_samples:
            ig = integrated_gradients(model_bilstm, p12_X[idx], target_class_idx=s, m_steps=15)
            _, feat_pct = decompose_attributions(ig)
            attrs.append(feat_pct[feat_indices])
        state_feature_attrs[s] = np.mean(attrs, axis=0)

df_panel_b = pd.DataFrame(state_feature_attrs, index=CATEGORIES, columns=key_features)
sns.heatmap(df_panel_b.T, ax=axes[1], cmap="YlOrRd", annot=True, fmt=".1f", cbar_kws={'label': 'Mean Saliency (%)'})
axes[1].set_title("Panel B: State-Dependent Electrophysiological Feature Attributions Across Operational States", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Clinical Operational State", fontsize=11)
axes[1].set_ylabel("Morphological Feature", fontsize=11)

# Panel C: Spatial Derivation Attribution Across States
state_channel_attrs = np.zeros((5, NUM_CHANNELS))
for s in range(5):
    s_idx = np.where(p12_y == s)[0]
    if len(s_idx) > 0:
        s_samples = s_idx[:min(len(s_idx), 10)]
        attrs = []
        for idx in s_samples:
            ig = integrated_gradients(model_bilstm, p12_X[idx], target_class_idx=s, m_steps=15)
            ch_pct, _ = decompose_attributions(ig)
            attrs.append(ch_pct)
        state_channel_attrs[s] = np.mean(attrs, axis=0)

df_panel_c = pd.DataFrame(state_channel_attrs, index=CATEGORIES, columns=CHANNELS)
sns.heatmap(df_panel_c, ax=axes[2], cmap="Blues", annot=False, cbar_kws={'label': 'Saliency (%)'})
axes[2].set_title("Panel C: Spatial Derivation Attribution Dynamics Across Bipolar Derivations", fontsize=13, fontweight='bold')
axes[2].set_xlabel("EEG Derivation", fontsize=11)
axes[2].set_ylabel("Clinical Operational State", fontsize=11)
plt.setp(axes[2].get_xticklabels(), rotation=45, ha="right")

plt.tight_layout()
fig17_path = os.path.join(output_dir, "Figure_17_Patient12_XAI_Deep_Dive.png")
plt.savefig(fig17_path, dpi=300)
plt.show()
print(f"Saved Figure 17 (3-Panel Deep Dive) to: {fig17_path}")


In [ ]:
# ==============================================================================
# Cell 12: Distribution of Recording-Level Metrics (Figure 11 Equivalent for Staging)
# ==============================================================================

plt.figure(figsize=(14, 5), dpi=300)

metrics_to_plot = ["Exact Acc. (%)", "CTA (<=1 Step %)", "MTSD (Steps)", "Macro F1"]
plot_colors = ["#1f77b4", "#2ca02c", "#d62728", "#9467bd"]

for i, m in enumerate(metrics_to_plot, 1):
    plt.subplot(1, 4, i)
    sns.boxplot(y=df_table_10[m], color=plot_colors[i-1], width=0.4, boxprops=dict(alpha=0.7))
    sns.stripplot(y=df_table_10[m], color="black", jitter=0.2, size=6, alpha=0.8)
    plt.title(m, fontsize=12, fontweight='bold')
    plt.ylabel("")

plt.suptitle("Recording-Level Metric Distributions Across 21 Independent Test Sessions (Bi-LSTM)", fontsize=14, fontweight='bold')
plt.tight_layout()
dist_path = os.path.join(output_dir, "Figure_PatientWise_Staging_Distributions.png")
plt.savefig(dist_path, dpi=300)
plt.show()
print(f"Saved Metric Distributions Plot to: {dist_path}")
